# **ZERO-SHOT CLASSIFICATION FOR LLAMA-3.2-1B**


In [ ]:
# load test dataset, retrieve this from the .csv file

import pandas as pd

test_frame = pd.read_csv('fpb_test.csv')

text_col = 'text'  # text
label_col = 'label' # label
source_col = 'source' # sure why not
unique_labels = test_frame[label_col].unique().tolist()

print(f"loaded {len(test_frame)} rows.")
print(f"labels to test: {unique_labels}")

display(test_frame)

loaded 727 rows.
labels to test: ['Neutral', 'Fear', 'Optimism', 'Sadness', 'Joy']


,text,label,source,clean_text
0,the share capital of alma media corporation bu...,Neutral,FinancialPhraseBank,the share capital of alma media corporation bu...
1,the eu commission said earlier it had fined th...,Fear,FinancialPhraseBank,the eu commission said earlier it had fined th...
2,"kesko pursues a strategy of healthy , focused ...",Optimism,FinancialPhraseBank,kesko pursues a strategy of healthy focused gr...
3,down to eur5 .9 m h1 '09 3 august 2009 - finni...,Sadness,FinancialPhraseBank,down to eurNUM NUM m hNUM NUM NUM august NUM f...
4,"cencorp would focus on the development , manuf...",Neutral,FinancialPhraseBank,cencorp would focus on the development manufac...
...,...,...,...,...
722,7 march 2011 - finnish it company digia oyj he...,Optimism,FinancialPhraseBank,NUM march NUM finnish it company digia oyj hel...
723,"operating loss totaled eur 0.8 mn , compared t...",Sadness,FinancialPhraseBank,operating loss totaled eur NUM NUM mn compared...
724,aspocomp has repaid its interest bearing liabi...,Optimism,FinancialPhraseBank,aspocomp has repaid its interest bearing liabi...
725,the contract also includes cutting and edging ...,Neutral,FinancialPhraseBank,the contract also includes cutting and edging ...


In [ ]:
# log into huggingface

import os
import sys

google_colab = "google.colab" in sys.modules and not os.environ.get("VERTEX_PRODUCT")

if google_colab:
    # Use secret if running in Google Colab
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
else:
    # Store Hugging Face data under `/content` if running in Colab Enterprise
    if os.environ.get("VERTEX_PRODUCT") == "COLAB_ENTERPRISE":
        os.environ["HF_HOME"] = "/content/hf"
    # Authenticate with Hugging Face
    from huggingface_hub import get_token
    if get_token() is None:
        from huggingface_hub import notebook_login
        notebook_login()

In [ ]:
!pip install -q transformers accelerate bitsandbytes scikit-learn

import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# configure model
model_id = "meta-llama/Llama-3.2-1B"

# load model and tokeniser
tokenizer = AutoTokenizer.from_pretrained(model_id)

# load in 4-bit to standard colab gpu memory
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config
)

# classify function (zero-shot, no samples)
def classify(text, candidate_labels):

    prompt = f"Categories of text: {', '.join(candidate_labels)}.\n"

    prompt += f"Classify the text below into one out of the five categories above."

    prompt += f"Text: ' {text}'\nCategory: "

    #print(prompt)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=50, do_sample=False, pad_token_id=tokenizer.eos_token_id)

    # Decode only the newly generated tokens, skipping the input prompt
    generated_token_ids = output[0][inputs.input_ids.shape[1]:]
    generated_text = tokenizer.decode(generated_token_ids, skip_special_tokens=True).strip()

    predicted_label = "Unknown" # default value

    # remove any "category" labels that may be left in the generated text
    cleaned_generated_text = generated_text.replace("Category: ", "").strip()

    # case-insensitive matching
    generated_text_lower = cleaned_generated_text.lower()

    # Attempt to find the best match by checking if the label is *in* the generated text
    for label in candidate_labels:
        if label.lower() in generated_text_lower:
            predicted_label = label
            break

    return predicted_label

# time to evaluate!
y_true = test_frame[label_col].tolist() # all the correct labels
y_pred = [classify(text, unique_labels) for text in test_frame[text_col]] # all the predicted labels

# basic metrics
accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')

# export metrics
results_dict = {
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Value': [accuracy, precision, recall, f1]
}

# create dataframe
results_df = pd.DataFrame(results_dict)

# export to csv
results_df.to_csv('zero_shot_llama_01.csv', index=False)

print("Results saved to 'zero_shot_llama_01.csv'")
display(results_df)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Results saved to 'zero_shot_llama_01.csv'


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


,Metric,Value
0,Accuracy,0.587345
1,Precision,0.354872
2,Recall,0.587345
3,F1 Score,0.442429
